In [ ]:
print("Hello")

In [ ]:
!nvidia-smi

## Setup

In [ ]:
import torch
import torchvision
import torch.nn as nn
import numpy
import pandas as pd
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import torchvision.transforms as transforms
from PIL import Image

In [ ]:
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.__file__)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
print(torch.cuda.device_count())

In [ ]:
TRAIN_DIR = "./house/train/train/"
TEST_DIR = "./house/test/test/"
SAMPLE_SUBMISSION_CSV = "./house/sample_submission.csv"
TRAIN_CSV = "./house/train.csv"

## Anything go on

In [ ]:
model = torchvision.models.efficientnet_b0()
model

In [ ]:
model.classifier[1] = nn.Linear(in_features=1280, out_features=2, bias=True)
model

In [ ]:
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
sample_submission_df

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df['id'] = train_df['image_name']
train_df.head()

In [ ]:
test_df = sample_submission_df.copy()
test_df.head()

## Data

In [ ]:
train_transform = transforms.Compose([
	transforms.Resize((224, 224)),
	transforms.RandomHorizontalFlip(),
	transforms.ToTensor(),
	transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
	transforms.Resize((224, 224)),
	transforms.ToTensor(),
	transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class ImageDataset(Dataset):
	def __init__(self, df, img_dir, preprocess, isTest=False):
		self.df = df
		self.img_dir = img_dir
		self.preprocess = preprocess
		self.isTest = isTest

	def __len__(self):
		return len(self.df)

	def __getitem__(self, index):
		row = self.df.iloc[index]

		image_path = self.img_dir + row['id']
		if self.isTest:
			image_path += ".jpg"

		img = cv2.imread(image_path)

		img = Image.open(image_path).convert('RGB')

		img = self.preprocess(img)

		if self.isTest:
			label = 0
		else:
			label = torch.tensor(row['class'])
		return img, label, row['id']

In [ ]:
train, val = train_test_split(train_df, test_size=0.2)

In [ ]:
train.head()

In [ ]:
val.head()

In [ ]:
train_dataset = ImageDataset(train, TRAIN_DIR, train_transform)
val_dataset = ImageDataset(val, TRAIN_DIR, train_transform)
test_dataset = ImageDataset(test_df, TEST_DIR, test_transform, isTest=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Training

In [ ]:
lr = 1e-4
epochs = 5

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr)

model = model.to(device) 

for epoch in range(epochs):
	# Training
	model.train()
	train_loss = 0.0

	for images, labels, _ in tqdm(train_loader, desc=f'Epoch {epoch} - Training'):
		images = images.to(device)
		labels = labels.to(device)

		optimizer.zero_grad()

		outputs = model(images)
		loss = criterion(outputs, labels)
		
		loss.backward()

		optimizer.step()

		train_loss += loss.item() * images.size(0) 

	epoch_train_loss = train_loss / len(train_loader.dataset)

	# Eval
	model.eval()
	val_loss = 0.0

	with torch.no_grad():
		for images, labels, _ in tqdm(val_loader, desc=f'Epoch {epoch} - Evaluating'):
			images = images.to(device)
			labels = labels.to(device)

			outputs = model(images)
			loss = criterion(outputs, labels)

			val_loss += loss.item() * images.size(0)

	epoch_val_loss = val_loss / len(val_loader.dataset)

	print(f"Epoch {epoch}: train_loss: {epoch_train_loss:.4f} val_loss: {epoch_val_loss:.4f}")

## Inference

In [ ]:
model.eval()
pred = []
pred_ids = [] 

with torch.no_grad():
	for images, _, batch_ids in tqdm(test_loader, desc='Inference'):
		images = images.to(device)
		outputs = model(images)

		batch_preds = outputs.argmax(dim=1).cpu().numpy().tolist()

		pred.extend(batch_preds)
		pred_ids.extend(batch_ids)

In [ ]:
submission_df = test_df.copy()
submission_df['class'] = pred
submission_df = submission_df[['id', 'class']]
print(submission_df.head())

In [ ]:
submission_df.to_csv('submission.csv', index=False)